# II. Genre Identification #

> **Objective**: This notebook main goal is to create a classification model that can predict accurately the musical genre of a song, based on its lyrics

## Grid Search Implementation ##

> In this step the model will be trained. To do so, different combinations of feature extraction and classification models were experimented. 


**Feature Extraction**: 
- Bag of Words
- TF-IDF
- Word2Vec
- Doc2Vec

**Classification Models**:
- Logistic Regression
- Random Forest
- Neural Networks


**Step 1:** Importing the necessary libraries and the datasets.

In [5]:
import pandas as pd

# models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn import metrics
from sklearn.metrics import f1_score

# feature extraction
from gensim.models import Word2Vec, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_extraction import DictVectorizer

# model selection
from sklearn.model_selection import GridSearchCV, StratifiedKFold

from tqdm import tqdm
tqdm.pandas()

import utils


In [6]:
train_data = pd.read_csv('data/train_after_eda.csv')
train_data.head()

,id,title,tag,artist,views,lyrics,eda_lyrics
0,535805,Walk Away,rock,Tony Molina,699,When you said you loved me\nDid you mean it th...,said loved mean change mind one day pretend ca...
1,7519483,Gotta Make It Kid Naruto Rap,rap,Reece Lett,4,Kid Naruto Rap\n[Hook]\nEverybody wants you to...,kid naruto rap everybody wants hurt everybody ...
2,4892808,​this is what i asked for,pop,Elliot (DNK),389,[Verse 1]\nPeople tell me I've changed\nI find...,people tell changed find hard explain maybe fe...
3,1584150,Stealing Hearts,pop,Katie Armiger,126,You've been warned about me\nDon't try to get ...,warned try get close know want chain leave gho...
4,7639050,Get Ready,country,John Campbell Munro,2,[Verse 1]\nI can see the end is coming but I’v...,see end coming never liked odds need way even ...


In [7]:
train_data['cleaned_lyrics'] = train_data['lyrics'].apply(
    lambda text: utils.clean_lyrics(
        text,
        keep_hashtags = False,
        keep_hashtags_words = True,
        keep_emojis = False,
        keep_structural_content = False,
        keep_punctuation = False,
        lower_case = True,
        lemmatization = False,
        stemming = False,
        remove_stopwords = False,
        custom_stopwords = False,
        tokenized_output = False,
        sentence_output = False
    )
)

**Step 2:** Grid Search

> In this step, the objective is to find the best combination of stop word removal approach, Feature Extractor and Classification model

In [8]:
# Train Test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(train_data['cleaned_lyrics'], train_data['tag'], test_size=0.2, random_state=42)

In [9]:
import importlib
import utils
importlib.reload(utils)

models = ['LogisticRegression', 'RandomForest', 'NeuralNetwork']

vectorizer_names = ['DictVectorizer', 'CountVectorizer', 'TfidfVectorizer', 'Word2Vec', 'Doc2Vec']

stopword_options = ["none", "standard", "custom"]

combo, train_score, val_score = utils.grid_search(vectorizer_names, models, stopword_options, X_train, y_train)

KeyboardInterrupt: 

> Best combination found:

**Stop Word Approach:** custom

**Feature Extractor:** Doc2Vec

**Model:** Neural Network


**Step 3:** Perform default grid search using Doc2Vec as feature extractor and Neural Network as classification model.

In [ ]:
# # Tagging documents

# X_train_tokens = utils.tokenize_with_stopwords(sw_mode='custom', X=X_train)
# tagged_data = [TaggedDocument(words=doc, tags=[str(i)]) for i, doc in enumerate(X_train_tokens)]
# print("Number of tagged documents:", len(tagged_data))
# print("Sample tagged doc:", tagged_data[0] if len(tagged_data) > 0 else "EMPTY!")

# # Training Doc2Vec model
# model = Doc2Vec(vector_size=300, window=5, min_count=1, workers=4, epochs=20)
# model.build_vocab(tagged_data)
# model.train(tagged_data, total_examples=model.corpus_count, epochs=model.epochs)

# # Generate vectors for training and testing sets
# X_train_vectors = utils.infer_vectors(model, X_train_tokens)


# # Define the parameter grid
# param_grid = {
#     'hidden_layer_sizes': [(50,), (100,), (100, 50)],
#     'activation': ['relu', 'tanh'],
#     'solver': ['adam', 'sgd'],
#     'alpha': [0.0001, 0.001, 0.01],
#     'learning_rate_init': [0.001, 0.01]
# }

# # Initialize model
# mlp_model = MLPClassifier(max_iter=500, random_state=42, early_stopping=True)

# # Initialize StratifiedKFold for cross-validation
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# # Initialize GridSearchCV with F1 score as the scoring metric
# grid_search = GridSearchCV(
#     estimator=mlp_model, 
#     param_grid=param_grid, 
#     cv=cv, 
#     scoring='f1_weighted'
#     )

# # Fit the grid search to the data
# grid_search.fit(X_train_vectors, y_train)

# # Print the best hyperparameters found
# print("Best hyperparameters:", grid_search.best_params_)

# # Get the best model
# best_model = grid_search.best_estimator_

**Best hyperparameters:**

{'activation': 'tanh', 'alpha': 0.001, 'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.001, 'solver': 'adam'}

**Step 4:** Train the model

In [ ]:
# Data
X_tokens = utils.tokenize_with_stopwords(sw_mode='custom', X=train_data['eda_lyrics'])
y = train_data['tag']

# Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

overall_score_columns = ["accuracy","precision","recall","f1"]
overall_score_data = []
predictions_data = []

for train_index, val_index in skf.split(X_tokens, y):
    # Split train an validation data
    X_train = [X_tokens[i] for i in train_index]
    X_val = [X_tokens[i] for i in val_index]    

    y_train = y.iloc[train_index]
    y_val = y.iloc[val_index]

    # Tagging documents
    tagged_data = [TaggedDocument(words=doc, tags=[str(i)]) for i, doc in enumerate(X_train)]

    # Initialize the Doc2Vec model
    model = Doc2Vec(vector_size=300, window=5, min_count=1, workers=4, epochs=10)

    # Training Doc2Vec model
    model.build_vocab(tagged_data)
    model.train(tagged_data, total_examples=model.corpus_count, epochs=model.epochs)

    # Generate vectors for training and testing sets
    X_train_vectors = utils.infer_vectors(model, X_train)
    X_val_vectors = utils.infer_vectors(model, X_val)

    # Initialize and train the logistic regression model
    mlp_model = MLPClassifier(
    activation='tanh',           
    alpha=0.001,                
    hidden_layer_sizes=(100, 50), 
    learning_rate_init=0.001, 
    solver='adam',
    max_iter=500, 
    random_state=42, 
    early_stopping=True
    )
    mlp_model.fit(X_train_vectors, y_train)

    # Predictions on the test set
    y_pred = mlp_model.predict(X_val_vectors)

    fold_score = list(utils.fold_score_calculator(y_pred, y_val))
    overall_score_data.append(fold_score)
        
    # Store predictions and corresponding indices in a DataFrame
    fold_predictions = pd.DataFrame({
        'ValIndex': val_index,
        'Prediction': y_pred
        })
    predictions_data.append(fold_predictions)

scores_df = pd.DataFrame(overall_score_data, columns=overall_score_columns)
predictions_df = pd.concat(predictions_data, ignore_index=True)


**Step 5:** Print the results

In [ ]:
scores_df

,accuracy,precision,recall,f1
0,0.652534,0.625883,0.652534,0.620640
1,0.652014,0.628835,0.652014,0.622619
2,0.657764,0.635624,0.657764,0.616339
3,0.652496,0.628349,0.652496,0.618337
4,0.651334,0.627838,0.651334,0.619575
